# HELICoiD: SegFormer + MedFormer (image + SegFormer mask)

Built on `HSI_Final_Code.ipynb`, with two changes:

1. **Data:** the HELICoiD brain images and their expert labels, as tumour (1) vs not tumour (0). Pixels the experts did not label are set to 255 and ignored in training and scoring.
2. **MedFormer** is attached after SegFormer: the image together with the segmented output of SegFormer (the predicted mask) is the input to MedFormer, which is trained with the same labels.

Run on Google Colab with a GPU (Runtime → Change runtime type → T4 GPU), then Runtime → Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%pip install -q "transformers<5" einops timm

# MedFormer code from the authors' repository (same version as the MedFormer folder in the project)
!git clone -q https://github.com/yhygao/CBIM-Medical-Image-Segmentation.git
!git -C CBIM-Medical-Image-Segmentation checkout -q 7c26979
!mkdir -p /content/MedFormer && touch /content/MedFormer/__init__.py
!cd CBIM-Medical-Image-Segmentation/model/dim2 && cp medformer.py medformer_utils.py conv_layers.py trans_layers.py utils.py /content/MedFormer/

In [ ]:
import os
import numpy as np
import scipy.io as sio
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from skimage.segmentation import slic
from skimage.morphology import opening, closing, disk
from skimage.measure import label, regionprops
from skimage.color import label2rgb
import matplotlib.pyplot as plt
import cv2

In [ ]:
DATASET_DIR = '/content/drive/MyDrive/dataset_helicoid'
ZIP_NAMES = ['FirstCampaign.zip', 'SecondCampaign.zip', 'ThirdCampaign.zip']
N_BANDS = 128  # the 826 bands are averaged down to 128

In [ ]:
import glob
import shutil
import zipfile

ENVI_DTYPES = {1: np.uint8, 2: np.int16, 3: np.int32, 4: np.float32, 5: np.float64, 12: np.uint16}

def read_envi(hdr_path):
    """Reads an ENVI image (.hdr + data file) into an array of shape (lines, samples, bands)."""
    header = dict(line.split('=', 1) for line in open(hdr_path) if '=' in line and '{' not in line)
    header = {k.strip(): v.strip() for k, v in header.items()}
    samples, lines, bands = int(header['samples']), int(header['lines']), int(header['bands'])
    dtype = np.dtype(ENVI_DTYPES[int(header['data type'])])
    if header.get('byte order') == '1':
        dtype = dtype.newbyteorder('>')
    data_file = next(f for f in [hdr_path[:-4], hdr_path[:-4] + '.raw', hdr_path[:-4] + '.img'] if os.path.isfile(f))
    data = np.fromfile(data_file, dtype=dtype, count=samples * lines * bands,
                       offset=int(header.get('header offset', 0)))
    interleave = header.get('interleave', 'bsq').lower()
    if interleave == 'bsq':
        return data.reshape(bands, lines, samples).transpose(1, 2, 0)
    if interleave == 'bil':
        return data.reshape(lines, bands, samples).transpose(0, 2, 1)
    return data.reshape(lines, samples, bands)

def load_helicoid_image(image_zip):
    """Returns the calibrated hyperspectral cube and the expert label map of one HELICoiD image."""
    with zipfile.ZipFile(image_zip) as z:
        z.extractall('tmp_image')
    hdrs = glob.glob('tmp_image/**/*.hdr', recursive=True)
    find = lambda name: next(h for h in hdrs if os.path.basename(h).lower().startswith(name))

    cube = read_envi(find('raw')).astype(np.float32)
    white = read_envi(find('white')).astype(np.float32).mean(axis=0)
    dark = read_envi(find('dark')).astype(np.float32).mean(axis=0)
    gt = read_envi(find('gt'))[:, :, 0].astype(np.uint8)
    shutil.rmtree('tmp_image')
    if white.shape[0] != cube.shape[1]:  # reference of a different width: one value per band
        white, dark = white.mean(axis=0), dark.mean(axis=0)

    # Calibration: reflectance = (raw - dark) / (white - dark)
    cube -= dark
    cube /= np.maximum(white - dark, 1e-6)
    np.clip(cube, 0, 2, out=cube)
    return cube, gt

def bin_bands(cube, n_bands):
    """Averages neighbouring bands so the cube has n_bands bands."""
    groups = np.array_split(np.arange(cube.shape[2]), n_bands)
    return np.stack([cube[:, :, g].mean(axis=2) for g in groups], axis=2)

In [ ]:
def normalize_hsi_cube(hsi):
    """Normalize each band individually to [0, 1] range."""
    hsi_norm = np.zeros_like(hsi)
    for i in range(hsi.shape[2]):
        band = hsi[:, :, i]
        band_min, band_max = np.min(band), np.max(band)
        hsi_norm[:, :, i] = (band - band_min) / (band_max - band_min + 1e-8)
    return hsi_norm

In [ ]:
def apply_pca(hsi_cube, n_components=3):
    h, w, d = hsi_cube.shape
    reshaped = hsi_cube.reshape(-1, d)
    pca = PCA(n_components=n_components)
    pca_data = pca.fit_transform(reshaped)
    return pca_data.reshape(h, w, n_components)

In [ ]:
from PIL import Image

MASK_SAVE_DIR = 'TrainData/masks'
IMAGE_SAVE_DIR = 'TrainData/images'
os.makedirs(MASK_SAVE_DIR, exist_ok=True)
os.makedirs(IMAGE_SAVE_DIR, exist_ok=True)

for zip_name in ZIP_NAMES:
    print(f"Copying {zip_name} from Google Drive...")
    shutil.copy(os.path.join(DATASET_DIR, zip_name), zip_name)
    with zipfile.ZipFile(zip_name) as campaign_zip:
        for image_zip in [n for n in campaign_zip.namelist() if n.endswith('.zip')]:
            path = campaign_zip.extract(image_zip, 'tmp')
            hsi_cube, gt = load_helicoid_image(path)
            os.remove(path)

            hsi_cube = bin_bands(hsi_cube, N_BANDS)
            hsi_cube = np.nan_to_num(hsi_cube)
            hsi_cube = normalize_hsi_cube(hsi_cube)
            pca_rgb = apply_pca(hsi_cube)

            # Expert labels 1 normal, 2 tumour, 3 blood vessel, 4 background, 0 unlabelled
            # -> 1 tumour, 0 not tumour, 255 unlabelled (ignored)
            mask = np.full(gt.shape, 255, dtype=np.uint8)
            mask[gt > 0] = 0
            mask[gt == 2] = 1

            vis_rgb = ((pca_rgb - pca_rgb.min()) / (pca_rgb.max() - pca_rgb.min()) * 255).astype(np.uint8)

            image_name = zip_name[:-4] + '__' + os.path.basename(image_zip).replace('.zip', '.png')
            Image.fromarray(vis_rgb).save(os.path.join(IMAGE_SAVE_DIR, image_name))
            Image.fromarray(mask).save(os.path.join(MASK_SAVE_DIR, image_name))
    os.remove(zip_name)

print("Images:", len(os.listdir(IMAGE_SAVE_DIR)))
print("Masks:", len(os.listdir(MASK_SAVE_DIR)))

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation
from transformers import TrainingArguments, Trainer
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
MASK_SAVE_DIR = 'TrainData/masks'
IMAGE_SAVE_DIR = 'TrainData/images'
RESULTS_DIR = 'SegformerResults'
VISUALIZATION_DIR = 'Visualizations'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(VISUALIZATION_DIR, exist_ok=True)

class HSIImageDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, feature_extractor=None):
        self.image_paths = sorted(glob.glob(os.path.join(image_dir, "*.png")))
        self.mask_paths = sorted(glob.glob(os.path.join(mask_dir, "*.png")))
        self.transform = transform
        self.feature_extractor = feature_extractor

        assert len(self.image_paths) == len(self.mask_paths), f"Mismatch: {len(self.image_paths)} images vs {len(self.mask_paths)} masks"
        print(f"Dataset initialized with {len(self.image_paths)} samples")

    def __getitem__(self, idx):
        # Load image and mask
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")  # Grayscale

        image_np = np.array(image)
        mask_np = np.array(mask)

        if self.transform:
            transformed = self.transform(image=image_np, mask=mask_np)
            image_tensor = transformed["image"]
            mask_tensor = transformed["mask"]
        else:
            image_tensor = T.ToTensor()(image)
            mask_tensor = torch.from_numpy(mask_np)

        # Ensure mask is the right type and shape
        if len(mask_tensor.shape) == 2:  # [H, W]
            mask_tensor = mask_tensor.long()
        elif len(mask_tensor.shape) == 3:  # [C, H, W] - squeeze if single channel
            if mask_tensor.shape[0] == 1:
                mask_tensor = mask_tensor.squeeze(0).long()
            else:
                mask_tensor = mask_tensor.long()

        return {
            "pixel_values": image_tensor.float(),
            "labels": mask_tensor
        }

    def __len__(self):
        return len(self.image_paths)

def compute_comprehensive_metrics(eval_pred):
    """Comprehensive evaluation metrics for semantic segmentation."""
    predictions, labels = eval_pred

    # Debug print to understand shapes
    print(f"Predictions shape: {predictions.shape}")
    print(f"Labels shape: {labels.shape}")

    # Handle the predictions - they come as logits
    if len(predictions.shape) == 4:  # [batch, num_classes, height, width]
        predictions = np.argmax(predictions, axis=1)  # [batch, height, width]
    elif len(predictions.shape) == 3 and predictions.shape[-1] == 2:  # [batch, height, width, num_classes]
        predictions = np.argmax(predictions, axis=-1)  # [batch, height, width]

    # Ensure labels and predictions have the same shape
    if predictions.shape != labels.shape:
        print(f"Shape mismatch - Predictions: {predictions.shape}, Labels: {labels.shape}")

        # Try to resize predictions to match labels
        from scipy.ndimage import zoom
        if len(predictions.shape) == 3 and len(labels.shape) == 3:
            # Calculate zoom factors for each dimension
            zoom_factors = [labels.shape[i] / predictions.shape[i] for i in range(3)]
            predictions = zoom(predictions, zoom_factors, order=0)  # Nearest neighbor for labels
            print(f"Resized predictions to: {predictions.shape}")

    # Flatten for metrics computation
    predictions_flat = predictions.flatten()
    labels_flat = labels.flatten()

    # Ignore unlabelled pixels (255)
    valid = labels_flat != 255
    predictions_flat, labels_flat = predictions_flat[valid], labels_flat[valid]

    # Ensure same length after flattening
    min_length = min(len(predictions_flat), len(labels_flat))
    predictions_flat = predictions_flat[:min_length]
    labels_flat = labels_flat[:min_length]

    print(f"Final flattened shapes - Predictions: {len(predictions_flat)}, Labels: {len(labels_flat)}")

    # Basic metrics
    try:
        accuracy = accuracy_score(labels_flat, predictions_flat)

        # For binary segmentation (background=0, lesion=1)
        precision = precision_score(labels_flat, predictions_flat, average='binary', zero_division=0)
        recall = recall_score(labels_flat, predictions_flat, average='binary', zero_division=0)
        f1 = f1_score(labels_flat, predictions_flat, average='binary', zero_division=0)

        # IoU (Jaccard Index)
        iou = jaccard_score(labels_flat, predictions_flat, average='binary', zero_division=0)

        # Dice Score (F1 score is equivalent to Dice for binary classification)
        dice = f1

        # Specificity (True Negative Rate)
        tn = np.sum((labels_flat == 0) & (predictions_flat == 0))
        fp = np.sum((labels_flat == 0) & (predictions_flat == 1))
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

        # Sensitivity (same as recall)
        sensitivity = recall

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "iou": iou,
            "dice_score": dice,
            "specificity": specificity,
            "sensitivity": sensitivity
        }
    except Exception as e:
        print(f"Error computing metrics: {e}")
        return {
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1_score": 0.0,
            "iou": 0.0,
            "dice_score": 0.0,
            "specificity": 0.0,
            "sensitivity": 0.0
        }

def visualize_predictions(model, dataset, device, num_samples=8, save_path=None):
    """Visualize model predictions vs ground truth."""
    model.eval()

    # Randomly select samples
    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    fig, axes = plt.subplots(3, len(indices), figsize=(3*len(indices), 9))
    if len(indices) == 1:
        axes = axes.reshape(-1, 1)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            sample = dataset[idx]
            image = sample["pixel_values"].unsqueeze(0).to(device)
            true_mask = sample["labels"].numpy()

            # Get prediction
            outputs = model(image)
            predictions = outputs.logits

            # Process prediction
            predicted_mask = torch.argmax(predictions.squeeze(), dim=0).cpu().numpy()

            # Convert image for visualization
            img_vis = sample["pixel_values"].permute(1, 2, 0).numpy()
            img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())

            # Plot original image
            axes[0, i].imshow(img_vis)
            axes[0, i].set_title(f'Original Image {idx}')
            axes[0, i].axis('off')

            # Plot ground truth mask
            axes[1, i].imshow(np.where(true_mask == 255, 0.5, true_mask), cmap='gray', vmin=0, vmax=1)  # unlabelled in grey
            axes[1, i].set_title('Ground Truth')
            axes[1, i].axis('off')

            # Plot predicted mask
            axes[2, i].imshow(predicted_mask, cmap='gray')
            axes[2, i].set_title('Prediction')
            axes[2, i].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Visualization saved to {save_path}")
    plt.show()

In [ ]:
def visualize_overlay_predictions(model, dataset, device, num_samples=4, save_path=None):
    """Visualize predictions as colored overlays on original images."""
    model.eval()

    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    fig, axes = plt.subplots(2, len(indices), figsize=(4*len(indices), 8))
    if len(indices) == 1:
        axes = axes.reshape(-1, 1)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            sample = dataset[idx]
            image = sample["pixel_values"].unsqueeze(0).to(device)
            true_mask = sample["labels"].numpy()

            # Get prediction
            outputs = model(image)
            predictions = outputs.logits
            predicted_mask = torch.argmax(predictions.squeeze(), dim=0).cpu().numpy()

            # Print shapes for debugging
            print(f"Sample {i}:")
            print(f"  Image shape: {sample['pixel_values'].shape}")
            print(f"  True mask shape: {true_mask.shape}")
            print(f"  Predicted mask shape: {predicted_mask.shape}")

            # Convert image for visualization
            img_vis = sample["pixel_values"].permute(1, 2, 0).numpy()

            # Handle case where image has only 3 channels after PCA
            if img_vis.shape[2] == 3:
                # Normalize for RGB-like display
                img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())
            else:
                # If more than 3 channels, take first 3 or create RGB
                if img_vis.shape[2] > 3:
                    img_vis = img_vis[:, :, :3]
                else:
                    # If less than 3, replicate channels
                    img_vis = np.repeat(img_vis, 3, axis=2)[:, :, :3]
                img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())

            # Resize masks to match image dimensions if needed
            target_height, target_width = img_vis.shape[:2]

            if true_mask.shape != (target_height, target_width):
                print(f"  Resizing true_mask from {true_mask.shape} to ({target_height}, {target_width})")
                true_mask = cv2.resize(true_mask.astype(np.uint8),
                                     (target_width, target_height),
                                     interpolation=cv2.INTER_NEAREST).astype(bool)

            if predicted_mask.shape != (target_height, target_width):
                print(f"  Resizing predicted_mask from {predicted_mask.shape} to ({target_height}, {target_width})")
                predicted_mask = cv2.resize(predicted_mask.astype(np.uint8),
                                          (target_width, target_height),
                                          interpolation=cv2.INTER_NEAREST).astype(bool)

            # Create overlays
            gt_overlay = img_vis.copy()
            pred_overlay = img_vis.copy()

            # Ground truth overlay (green) - ensure boolean indexing works
            try:
                gt_overlay[true_mask == 1] = [0, 1, 0]
            except IndexError as e:
                print(f"Error with ground truth overlay: {e}")
                print(f"gt_overlay shape: {gt_overlay.shape}, true_mask shape: {true_mask.shape}")
                continue

            # Prediction overlay (red) - ensure boolean indexing works
            try:
                pred_overlay[predicted_mask == 1] = [1, 0, 0]
            except IndexError as e:
                print(f"Error with prediction overlay: {e}")
                print(f"pred_overlay shape: {pred_overlay.shape}, predicted_mask shape: {predicted_mask.shape}")
                continue

            # Plot ground truth overlay
            axes[0, i].imshow(gt_overlay)
            axes[0, i].set_title(f'Ground Truth Overlay {idx}')
            axes[0, i].axis('off')

            # Plot prediction overlay
            axes[1, i].imshow(pred_overlay)
            axes[1, i].set_title('Prediction Overlay')
            axes[1, i].axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Overlay visualization saved to {save_path}")
    plt.show()

In [ ]:
def plot_training_metrics(trainer, save_path=None):
    """Plot training and validation metrics over epochs."""

    # Extract metrics from trainer logs
    logs = trainer.state.log_history

    train_metrics = {}
    eval_metrics = {}

    for log in logs:
        if 'train_loss' in log:
            epoch = log.get('epoch', 0)
            if 'train_loss' not in train_metrics:
                train_metrics['train_loss'] = []
                train_metrics['epochs'] = []
            train_metrics['train_loss'].append(log['train_loss'])
            train_metrics['epochs'].append(epoch)

        if 'eval_loss' in log:
            epoch = log.get('epoch', 0)
            for key, value in log.items():
                if key.startswith('eval_'):
                    metric_name = key.replace('eval_', '')
                    if metric_name not in eval_metrics:
                        eval_metrics[metric_name] = []
                        if 'epochs' not in eval_metrics:
                            eval_metrics['epochs'] = []
                    eval_metrics[metric_name].append(value)
            if 'epochs' not in eval_metrics or len(eval_metrics['epochs']) < len(eval_metrics['loss']):
                eval_metrics['epochs'].append(epoch)

    # Create subplots
    num_metrics = len([k for k in eval_metrics.keys() if k != 'epochs'])
    fig, axes = plt.subplots(2, max(2, num_metrics//2), figsize=(15, 8))
    axes = axes.flatten()

    plot_idx = 0

    # Plot training and validation loss
    if 'train_loss' in train_metrics and 'loss' in eval_metrics:
        axes[plot_idx].plot(train_metrics['epochs'], train_metrics['train_loss'], 'b-', label='Train Loss')
        axes[plot_idx].plot(eval_metrics['epochs'], eval_metrics['loss'], 'r-', label='Val Loss')
        axes[plot_idx].set_title('Training and Validation Loss')
        axes[plot_idx].set_xlabel('Epoch')
        axes[plot_idx].set_ylabel('Loss')
        axes[plot_idx].legend()
        axes[plot_idx].grid(True)
        plot_idx += 1

    # Plot other metrics
    metric_names = ['accuracy', 'f1_score', 'iou', 'dice_score', 'precision', 'recall']

    for metric in metric_names:
        if metric in eval_metrics and plot_idx < len(axes):
            axes[plot_idx].plot(eval_metrics['epochs'], eval_metrics[metric], 'g-', marker='o')
            axes[plot_idx].set_title(f'Validation {metric.replace("_", " ").title()}')
            axes[plot_idx].set_xlabel('Epoch')
            axes[plot_idx].set_ylabel(metric.replace("_", " ").title())
            axes[plot_idx].grid(True)
            plot_idx += 1

    # Hide unused subplots
    for i in range(plot_idx, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Training metrics plot saved to {save_path}")
    plt.show()

In [ ]:
def evaluate_model_comprehensive(model, dataset, device):
    """Comprehensive evaluation of the trained model."""
    model.eval()

    all_predictions = []
    all_labels = []

    dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc="Evaluating")):
            images = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            # Print shapes for debugging (only first batch)
            if i == 0:
                print(f"Input images shape: {images.shape}")
                print(f"Labels shape: {labels.shape}")

            outputs = model(images)

            # Print output shape for debugging (only first batch)
            if i == 0:
                print(f"Model outputs shape: {outputs.logits.shape}")

            # Get predictions
            predictions = torch.argmax(outputs.logits, dim=1)

            # Print prediction shape for debugging (only first batch)
            if i == 0:
                print(f"Predictions shape: {predictions.shape}")
                print(f"Predictions flattened length: {predictions.cpu().numpy().flatten().shape}")
                print(f"Labels flattened length: {labels.cpu().numpy().flatten().shape}")

            # Handle potential shape mismatches
            if predictions.shape != labels.shape:
                print(f"Warning: Shape mismatch - Predictions: {predictions.shape}, Labels: {labels.shape}")

                # If predictions have extra dimensions, reshape appropriately
                if len(predictions.shape) > len(labels.shape):
                    predictions = predictions.squeeze()
                elif len(predictions.shape) < len(labels.shape):
                    # If labels have extra dimensions, reshape labels
                    labels = labels.squeeze()

                # If still mismatched, try interpolation (for different spatial resolutions)
                if predictions.shape != labels.shape:
                    print("Attempting to resize predictions to match labels...")
                    predictions = torch.nn.functional.interpolate(
                        predictions.float().unsqueeze(1),
                        size=labels.shape[-2:],
                        mode='nearest'
                    ).squeeze(1).long()

            # Flatten and add to lists
            pred_flat = predictions.cpu().numpy().flatten()
            label_flat = labels.cpu().numpy().flatten()

            all_predictions.extend(pred_flat)
            all_labels.extend(label_flat)

    # Convert to numpy arrays
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)

    # Ignore unlabelled pixels (255)
    valid = all_labels != 255
    all_predictions, all_labels = all_predictions[valid], all_labels[valid]

    print(f"Final - Predictions shape: {all_predictions.shape}")
    print(f"Final - Labels shape: {all_labels.shape}")

    # Ensure both arrays have the same length
    if len(all_predictions) != len(all_labels):
        min_length = min(len(all_predictions), len(all_labels))
        all_predictions = all_predictions[:min_length]
        all_labels = all_labels[:min_length]
        print(f"Truncated to common length: {min_length}")

    # Compute metrics
    metrics = {
        'accuracy': accuracy_score(all_labels, all_predictions),
        'precision': precision_score(all_labels, all_predictions, average='binary', zero_division=0),
        'recall': recall_score(all_labels, all_predictions, average='binary', zero_division=0),
        'f1_score': f1_score(all_labels, all_predictions, average='binary', zero_division=0),
        'iou': jaccard_score(all_labels, all_predictions, average='binary', zero_division=0),
        'specificity': precision_score(all_labels, all_predictions, pos_label=0, average='binary', zero_division=0)
    }

    metrics['dice_score'] = metrics['f1_score']  # Dice is equivalent to F1 for binary
    metrics['sensitivity'] = metrics['recall']   # Sensitivity is recall

    return metrics

In [ ]:
def custom_data_collator(batch):
    """Custom data collator to handle batching for segmentation."""
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }

In [ ]:
import time
from transformers import TrainerCallback

class EpochTimer(TrainerCallback):
    """Records the training time of each epoch."""
    def __init__(self):
        self.epoch_times = []

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.start = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        self.epoch_times.append(time.time() - self.start)

In [ ]:
print("Starting SegFormer training with comprehensive evaluation...")

# Check if data exists
if not os.path.exists(IMAGE_SAVE_DIR) or not os.path.exists(MASK_SAVE_DIR):
    print("Error: Training data directories not found. Please run the HSI processing code first.")

print(f"Images: {len(os.listdir(IMAGE_SAVE_DIR))}")
print(f"Masks: {len(os.listdir(MASK_SAVE_DIR))}")

# Initialize feature extractor and model with updated API
try:
    # Try newer API first
    from transformers import SegformerImageProcessor
    feature_extractor = SegformerImageProcessor(do_reduce_labels=False, size=256, resample=2)
except ImportError:
    # Fall back to older API
    feature_extractor = SegformerFeatureExtractor(do_reduce_labels=False, size=256, resample=2)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",  # Smaller model for faster training
    num_labels=2,  # background and lesion
    ignore_mismatched_sizes=True
)

# Set up transforms - simplified to avoid issues
transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.3),
    A.VerticalFlip(p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet normalization
    ToTensorV2()
])

# Create dataset
dataset = HSIImageDataset(IMAGE_SAVE_DIR, MASK_SAVE_DIR, transform=transform, feature_extractor=feature_extractor)

# Test a single sample to check shapes
sample = dataset[0]
print(f"Sample image shape: {sample['pixel_values'].shape}")
print(f"Sample label shape: {sample['labels'].shape}")
print(f"Sample label unique values: {torch.unique(sample['labels'])}")

# Train-validation split by patient, so no patient is in both sets
patients = np.array([os.path.basename(p).split('__')[1].split('-')[0] for p in dataset.image_paths])
train_patients, val_patients = train_test_split(np.unique(patients), test_size=0.2, random_state=42)
train_indices = np.where(np.isin(patients, train_patients))[0]
val_indices = np.where(np.isin(patients, val_patients))[0]

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset = torch.utils.data.Subset(dataset, val_indices)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Training arguments
args = TrainingArguments(
    output_dir=RESULTS_DIR,
    per_device_train_batch_size=2,  # Reduced for stability
    per_device_eval_batch_size=2,
    num_train_epochs=20,  # Reduced for testing
    eval_strategy="epoch",  # Updated parameter name
    save_strategy="epoch",
    logging_steps=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    logging_dir=os.path.join(RESULTS_DIR, 'logs'),
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_score",
    greater_is_better=True,
    report_to=[],  # Empty list instead of None
    save_total_limit=2,
    dataloader_pin_memory=False,  # Disable pin memory to avoid potential issues
)

# Initialize trainer
segformer_timer = EpochTimer()
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_comprehensive_metrics,
    data_collator=custom_data_collator,  # Add custom data collator
    callbacks=[segformer_timer],
)

# Train the model
print("Starting training...")
trainer.train()

# Save the final model
model.save_pretrained(os.path.join(RESULTS_DIR, "final_model"))
feature_extractor.save_pretrained(os.path.join(RESULTS_DIR, "final_model"))

# Comprehensive evaluation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
print("\nPerforming comprehensive evaluation...")

# Evaluate on validation set
val_metrics = evaluate_model_comprehensive(model, val_dataset, device)

print("\nFinal Validation Metrics:")
print("=" * 40)
for metric, value in val_metrics.items():
    print(f"{metric.replace('_', ' ').title()}: {value:.4f}")

# Create visualizations
print("\nGenerating visualizations...")

# Training metrics plot
plot_training_metrics(trainer, save_path=os.path.join(VISUALIZATION_DIR, 'training_metrics.png'))

# Prediction visualizations
visualize_predictions(model, val_dataset, device, num_samples=6,
                        save_path=os.path.join(VISUALIZATION_DIR, 'predictions.png'))

visualize_overlay_predictions(model, val_dataset, device, num_samples=4,
                            save_path=os.path.join(VISUALIZATION_DIR, 'overlay_predictions.png'))

# Save metrics to file
with open(os.path.join(RESULTS_DIR, 'final_metrics.txt'), 'w') as f:
    f.write("Final Validation Metrics\n")
    f.write("=" * 30 + "\n")
    for metric, value in val_metrics.items():
        f.write(f"{metric.replace('_', ' ').title()}: {value:.4f}\n")

print(f"\nTraining completed! Results saved in {RESULTS_DIR}")
print(f"Visualizations saved in {VISUALIZATION_DIR}")

## SegFormer + MedFormer

1. The trained SegFormer segments each image.
2. The image together with the segmented output of SegFormer (the predicted mask) is the input to MedFormer.
3. MedFormer is trained with the same labels that were given to SegFormer.
4. The output of MedFormer is compared with those labels.

In [ ]:
from MedFormer.medformer import MedFormer
import pandas as pd

### Image and SegFormer output, the input to MedFormer

In [ ]:
# Same preprocessing as SegFormer, without the random flips
eval_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])
plain_dataset = HSIImageDataset(IMAGE_SAVE_DIR, MASK_SAVE_DIR, transform=eval_transform)

model.eval()
images, segformer_masks, labels = [], [], []
with torch.no_grad():
    for i in range(len(plain_dataset)):
        sample = plain_dataset[i]
        logits = model(sample["pixel_values"].unsqueeze(0).to(device)).logits
        logits = F.interpolate(logits, size=(256, 256), mode="bilinear", align_corners=False)
        segformer_masks.append(logits.argmax(dim=1).cpu())
        labels.append(sample["labels"])
        images.append(sample["pixel_values"])
segformer_masks = torch.cat(segformer_masks)  # [N, 256, 256], 0 = not tumour, 1 = tumour
labels = torch.stack(labels)                  # [N, 256, 256], 255 = unlabelled
images = torch.stack(images)                  # [N, 3, 256, 256], the PCA image given to SegFormer

# PCA image (3 channels) + SegFormer mask (1 channel)
medformer_inputs = torch.cat([images, segformer_masks.unsqueeze(1).float()], dim=1)  # [N, 4, 256, 256]
train_set = torch.utils.data.TensorDataset(medformer_inputs[train_indices], labels[train_indices])
val_set = torch.utils.data.TensorDataset(medformer_inputs[val_indices], labels[val_indices])
train_loader = DataLoader(train_set, batch_size=2, shuffle=True)
val_loader = DataLoader(val_set, batch_size=2)

### Train MedFormer

In [ ]:
# MedFormer 2D settings from the authors' configuration (config/acdc/medformer_2d.yaml)
medformer = MedFormer(4, 2, base_chan=32, map_size=3, conv_block='BasicBlock',
                      conv_num=[2, 0, 0, 0, 0, 0, 2, 2], trans_num=[0, 2, 2, 2, 2, 2, 0, 0],
                      num_heads=[1, 4, 8, 16, 8, 4, 1, 1], fusion_depth=2, fusion_dim=512, fusion_heads=16,
                      expansion=2, attn_drop=0., proj_drop=0., proj_type='depthwise',
                      act=torch.nn.ReLU, aux_loss=False).to(device)
optimizer = torch.optim.AdamW(medformer.parameters(), lr=5e-4, weight_decay=0.05)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=255)  # unlabelled pixels are ignored

medformer_epoch_times = []
for epoch in range(20):
    start = time.time()
    medformer.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(medformer(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    medformer_epoch_times.append(time.time() - start)

    medformer.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            val_loss += loss_fn(medformer(x.to(device)), y.to(device)).item()
    print(f"Epoch {epoch + 1}: train loss {train_loss / len(train_loader):.4f}, "
          f"val loss {val_loss / len(val_loader):.4f}, time {medformer_epoch_times[-1]:.1f} s")

### Results: comparison with the labels (validation set)

In [ ]:
medformer.eval()
medformer_masks = []
with torch.no_grad():
    for x, _ in val_loader:
        medformer_masks.append(medformer(x.to(device)).argmax(dim=1).cpu())
medformer_masks = torch.cat(medformer_masks)

def segmentation_metrics(preds, labels):
    p, l = preds.numpy().flatten(), labels.numpy().flatten()
    valid = l != 255  # ignore unlabelled pixels
    p, l = p[valid], l[valid]
    tn = np.sum((l == 0) & (p == 0))
    fp = np.sum((l == 0) & (p == 1))
    return {
        "accuracy": accuracy_score(l, p),
        "precision": precision_score(l, p, zero_division=0),
        "recall": recall_score(l, p, zero_division=0),
        "dice_score": f1_score(l, p, zero_division=0),
        "iou": jaccard_score(l, p, zero_division=0),
        "specificity": tn / (tn + fp),
    }

val_labels = labels[val_indices]
results = pd.DataFrame({
    "SegFormer": segmentation_metrics(segformer_masks[val_indices], val_labels),
    "SegFormer + MedFormer": segmentation_metrics(medformer_masks, val_labels),
}).T
results.to_csv(os.path.join(RESULTS_DIR, "segformer_medformer_metrics.csv"))
results.round(4)

### Training time per epoch

In [ ]:
epoch_times = pd.DataFrame({
    "SegFormer (s)": pd.Series(segformer_timer.epoch_times),
    "MedFormer (s)": pd.Series(medformer_epoch_times),
})
epoch_times.index = epoch_times.index + 1
epoch_times.index.name = "Epoch"
epoch_times.to_csv(os.path.join(RESULTS_DIR, "epoch_times.csv"))

segformer_total = sum(segformer_timer.epoch_times) / 60
medformer_total = sum(medformer_epoch_times) / 60
print(f"Total training time: SegFormer {segformer_total:.1f} min, MedFormer {medformer_total:.1f} min, "
      f"together {segformer_total + medformer_total:.1f} min")
epoch_times.round(1)

### Examples (validation set)

In [ ]:
# Validation images with the most tumour pixels; unlabelled pixels are shown in grey
tumour_pixels = (val_labels == 1).sum(dim=(1, 2))
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for row, k in enumerate(torch.argsort(tumour_pixels, descending=True)[:4].tolist()):
    i = val_indices[k]
    image = Image.open(plain_dataset.image_paths[i]).resize((256, 256))
    label = np.where(labels[i] == 255, 0.5, labels[i])
    panels = [(image, "Image"), (label, "Label"), (segformer_masks[i], "SegFormer"),
              (medformer_masks[k], "SegFormer + MedFormer")]
    for col, (img, title) in enumerate(panels):
        axes[row, col].imshow(img, cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(VISUALIZATION_DIR, "segformer_medformer_predictions.png"), dpi=150)
plt.show()

### Save the results to Google Drive

In [ ]:
shutil.copytree(RESULTS_DIR, '/content/drive/MyDrive/HELICoiD_SegFormer_MedFormer_ImageMask_results/' + RESULTS_DIR,
                ignore=shutil.ignore_patterns('checkpoint-*'), dirs_exist_ok=True)
shutil.copytree(VISUALIZATION_DIR, '/content/drive/MyDrive/HELICoiD_SegFormer_MedFormer_ImageMask_results/' + VISUALIZATION_DIR,
                dirs_exist_ok=True)
print("Saved to My Drive > HELICoiD_SegFormer_MedFormer_ImageMask_results")